In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
print("All libraries version")
print("pandas version:",pd.__version__)
print("sqlite3 version:",sqlite3.sqlite_version)
print("matplotlib version:",plt.matplotlib.__version__)


All libraries version
pandas version: 2.2.2
sqlite3 version: 3.37.2
matplotlib version: 3.10.0


In [ ]:
import pandas as pd
print("successfully imported library")
df=pd.read_csv("student_performance.csv")
df.head()

successfully imported library


,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year
0,1001,Aarav Sharma,19,Male,Computer Science,2,85,78,72,91,92,Mumbai,2023
1,1002,Priya Patel,20,Female,Computer Science,2,76,82,88,79,87,Ahmedabad,2023
2,1003,Rohit Verma,19,Male,Electronics,2,65,74,61,55,78,Delhi,2023
3,1004,Sneha Reddy,20,Female,Mechanical,2,70,80,75,48,95,Hyderabad,2023
4,1005,Arjun Nair,19,Male,Computer Science,2,92,88,81,95,90,Kochi,2023


In [ ]:
print(df.shape)
print(df.shape[0])
print(df.shape[1])

(30, 13)
30
13


In [ ]:
if 'conn' in locals() and conn:
    conn.close()
    print("Closed existing database connection.")

conn=sqlite3.connect("college.db")
c=conn.cursor()
df.to_sql('students',
          conn,
          if_exists='replace',
          index=False   #do not write dataframe row number as a column
          )
c.execute("select count(*) from students")
count=c.fetchone()[0]
print("Database 'college.db' created/reconnected successfully")
print("table 'students' has ",count,"rows")

Database 'college.db' created/reconnected successfully
table 'students' has  30 rows


In [ ]:
#PRAGMA is a special sqlite command not standard sql
#table_info() shows the structure of the table
c.execute("pragma table_info(students)")
info=c.fetchall()
for col in info:
  print(col[1],col[2])

student_id INTEGER
name TEXT
age INTEGER
gender TEXT
department TEXT
semester INTEGER
math_score INTEGER
science_score INTEGER
english_score INTEGER
programming_score INTEGER
attendance_percentage INTEGER
city TEXT
admission_year INTEGER


In [ ]:
def run_query(sql, description=""):

    if description:
        print("=" * 50)
        print(description)
        print("=" * 50)

    result = pd.read_sql_query(sql, conn)

    print(result.to_string(index=False))

    return result

print("Helper function run_query defined successfully")

Helper function run_query defined successfully


In [ ]:
q="select student_id,name,department,math_score,attendance_percentage from students limit 10"
res1=run_query(q,"query1 : first 10 students(select+limit)")


query1 : first 10 students(select+limit)
 student_id         name       department  math_score  attendance_percentage
       1001 Aarav Sharma Computer Science          85                     92
       1002  Priya Patel Computer Science          76                     87
       1003  Rohit Verma      Electronics          65                     78
       1004  Sneha Reddy       Mechanical          70                     95
       1005   Arjun Nair Computer Science          92                     90
       1006  Meera Joshi      Electronics          58                     72
       1007  Kiran Kumar            Civil          73                     85
       1008  Divya Singh Computer Science          88                     96
       1009 Rahul Mishra       Mechanical          62                     68
       1010   Ananya Das Computer Science          95                     98


In [ ]:
query = """
SELECT rowid, name, math_score, science_score,
       programming_score, english_score
FROM students
ORDER BY rowid ASC
LIMIT 5 OFFSET 25;
"""
res1 = run_query(
    query,
    "Query 2 : Last 5 students records (SELECT + LIMIT)"
)

Query 2 : Last 5 students records (SELECT + LIMIT)
 rowid           name  math_score  science_score  programming_score  english_score
    26     Rekha Nair          72             77                 63             73
    27  Gaurav Shukla          84             79                 87             75
    28  Sunita Pillai          60             65                 39             68
    29     Nitin Jain          75             76                 50             70
    30 Akanksha Yadav          91             93                 94             87


In [ ]:
query3="SELECT name,department,math_score from Students order by math_score desc limit 5"
res1 = run_query(
    query,
    "Query 3 : first 5 top  students records"
)

Query 3 : first 5 top  students records
 rowid           name  math_score  science_score  programming_score  english_score
    26     Rekha Nair          72             77                 63             73
    27  Gaurav Shukla          84             79                 87             75
    28  Sunita Pillai          60             65                 39             68
    29     Nitin Jain          75             76                 50             70
    30 Akanksha Yadav          91             93                 94             87


In [ ]:
query4="SELECT name,department,programming_score from Students where programming_score between 50 and 75 order by programming_score desc"
res4 = run_query(
    query4,
    "Query 4 :   students records of programming s core 50 to 75"
)

Query 4 :   students records of programming s core 50 to 75
         name  department  programming_score
  Vikram Iyer Electronics                 72
 Ritu Agarwal Electronics                 69
   Rekha Nair Electronics                 63
Harish Pillai Electronics                 58
  Rohit Verma Electronics                 55
Preeti Saxena  Mechanical                 53
  Meera Joshi Electronics                 52
Kavya Nambiar  Mechanical                 51
   Nitin Jain  Mechanical                 50


In [ ]:
q5 = """
SELECT department,
       MAX(programming_score) AS top_programming_score
FROM students
WHERE department IN ('Electronics', 'Mechanical', 'Computer Science')
GROUP BY department order by programming_score desc limit 1;
"""

res5 = run_query(
    q5,
    "Query 5 : Department with top programming score"
)


Query 5 : Department with top programming score
      department  top_programming_score
Computer Science                     97


In [ ]:
q6 = """
SELECT department, name, attendance_percentage
FROM students
WHERE NOT attendance_percentage < 80;
"""

res6 = run_query(
    q6,
    "Query 6 : Excluding students below 80% attendance"
)

Query 6 : Excluding students below 80% attendance
      department           name  attendance_percentage
Computer Science   Aarav Sharma                     92
Computer Science    Priya Patel                     87
      Mechanical    Sneha Reddy                     95
Computer Science     Arjun Nair                     90
           Civil    Kiran Kumar                     85
Computer Science    Divya Singh                     96
Computer Science     Ananya Das                     98
     Electronics    Vikram Iyer                     83
           Civil    Pooja Gupta                     80
Computer Science     Suresh Rao                     88
      Mechanical  Kavya Nambiar                     91
     Electronics   Ritu Agarwal                     93
Computer Science Swati Kulkarni                     94
Computer Science   Nisha Kapoor                     89
Computer Science    Tanvi Mehta                     97
      Mechanical  Preeti Saxena                     86
Computer Scienc

In [ ]:
q6 = """
SELECT department, name, attendance_percentage
FROM students
WHERE attendance_percentage < 80;
"""

res6 = run_query(
    q6,
    "Query 6 : Excluding students below 80% attendance"
)

Query 6 : Excluding students below 80% attendance
      department           name  attendance_percentage
     Electronics    Rohit Verma                     78
     Electronics    Meera Joshi                     72
      Mechanical   Rahul Mishra                     68
Computer Science    Ajay Tiwari                     75
           Civil   Manoj Pandey                     65
      Mechanical Deepak Chauhan                     77
     Electronics  Harish Pillai                     74
           Civil   Sanjay Dubey                     70
           Civil  Sunita Pillai                     73


In [ ]:
dept_data={
    'dept_code':['CS','EC','ME',"CE"],
    'dept_name':['Computer Science','Electronics','Mechanical','Civil'],
    'hod':['Dr sharma','Dr Guru','Dr Hamlin','Dr Arjun'],
    'established':[1997,1996,1999,2000],
    'intake':[60,70,65,75]
}

df_dept=pd.DataFrame(dept_data)
df_dept



,dept_code,dept_name,hod,established,intake
0,CS,Computer Science,Dr sharma,1997,60
1,EC,Electronics,Dr Guru,1996,70
2,ME,Mechanical,Dr Hamlin,1999,65
3,CE,Civil,Dr Arjun,2000,75


In [ ]:
df_dept.to_sql('departments',
          conn,
          if_exists='replace',
          index=False   #do not write dataframe row number as a column
          )
print("created 'departments' table ")
print(df_dept.to_string(index=False))

dept_map={
    'Computer Science':'CS',
    'Electronics':'EC',
    'Mechanical':'ME',
    'Civil':'CE'
}

# Reload the original DataFrame to ensure 'department' column has full names
df = pd.read_csv("student_performance.csv")

# Create a new 'dept_code' column by mapping the 'department' names
df['dept_code'] = df['department'].map(dept_map)

# Save the DataFrame with the new 'dept_code' column to the 'students' table
df.to_sql('students',conn,if_exists='replace',index=False)
print("Updated 'students' table with 'dept_code' column.")

print("New schema for 'students' table after adding 'dept_code' column:")
c.execute("PRAGMA table_info(students)")
info = c.fetchall()
for col in info:
  print(col[1], col[2])

created 'departments' table 
dept_code        dept_name       hod  established  intake
       CS Computer Science Dr sharma         1997      60
       EC      Electronics   Dr Guru         1996      70
       ME       Mechanical Dr Hamlin         1999      65
       CE            Civil  Dr Arjun         2000      75
Updated 'students' table with 'dept_code' column.
New schema for 'students' table after adding 'dept_code' column:
student_id INTEGER
name TEXT
age INTEGER
gender TEXT
department TEXT
semester INTEGER
math_score INTEGER
science_score INTEGER
english_score INTEGER
programming_score INTEGER
attendance_percentage INTEGER
city TEXT
admission_year INTEGER
dept_code TEXT


After re-running the above cell, let's verify the schema of the `students` table to ensure the `dept_code` column is present:

In [ ]:
print("Current schema for 'students' table:")
c.execute("PRAGMA table_info(students)")
info = c.fetchall()
for col in info:
  print(col[1], col[2])

print("\nFirst 5 rows of the 'students' table with 'dept_code':")
students_with_dept_code = pd.read_sql_query("SELECT * FROM students LIMIT 5", conn)
display(students_with_dept_code)

Current schema for 'students' table:
student_id INTEGER
name TEXT
age INTEGER
gender TEXT
department TEXT
semester INTEGER
math_score INTEGER
science_score INTEGER
english_score INTEGER
programming_score INTEGER
attendance_percentage INTEGER
city TEXT
admission_year INTEGER
dept_code TEXT

First 5 rows of the 'students' table with 'dept_code':


,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year,dept_code
0,1001,Aarav Sharma,19,Male,Computer Science,2,85,78,72,91,92,Mumbai,2023,CS
1,1002,Priya Patel,20,Female,Computer Science,2,76,82,88,79,87,Ahmedabad,2023,CS
2,1003,Rohit Verma,19,Male,Electronics,2,65,74,61,55,78,Delhi,2023,EC
3,1004,Sneha Reddy,20,Female,Mechanical,2,70,80,75,48,95,Hyderabad,2023,ME
4,1005,Arjun Nair,19,Male,Computer Science,2,92,88,81,95,90,Kochi,2023,CS


If the `dept_code` column is now visible in the schema and the sample data, please proceed to re-run cell `p4c7uw-9nw5C`.

In [ ]:
query_join="""
SELECT
s.name,s.math_score,d.dept_name,d.hod,d.established from Students AS s INNER JOIN departments  AS d
ON s.dept_code=d.dept_code
ORDER BY s.math_score DESC
LIMIT 8"""

run_query(
    query_join)
q7="select * from students"
run_query(q7)

          name  math_score        dept_name       hod  established
    Ananya Das          95 Computer Science Dr sharma         1997
   Tanvi Mehta          93 Computer Science Dr sharma         1997
    Arjun Nair          92 Computer Science Dr sharma         1997
Akanksha Yadav          91 Computer Science Dr sharma         1997
Swati Kulkarni          90 Computer Science Dr sharma         1997
   Divya Singh          88 Computer Science Dr sharma         1997
  Ritu Agarwal          87      Electronics   Dr Guru         1996
     Amit Bose          86 Computer Science Dr sharma         1997
 student_id           name  age gender       department  semester  math_score  science_score  english_score  programming_score  attendance_percentage               city  admission_year dept_code
       1001   Aarav Sharma   19   Male Computer Science         2          85             78             72                 91                     92             Mumbai            2023        CS
       

,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year,dept_code
0,1001,Aarav Sharma,19,Male,Computer Science,2,85,78,72,91,92,Mumbai,2023,CS
1,1002,Priya Patel,20,Female,Computer Science,2,76,82,88,79,87,Ahmedabad,2023,CS
2,1003,Rohit Verma,19,Male,Electronics,2,65,74,61,55,78,Delhi,2023,EC
3,1004,Sneha Reddy,20,Female,Mechanical,2,70,80,75,48,95,Hyderabad,2023,ME
4,1005,Arjun Nair,19,Male,Computer Science,2,92,88,81,95,90,Kochi,2023,CS
5,1006,Meera Joshi,20,Female,Electronics,2,58,66,70,52,72,Pune,2023,EC
6,1007,Kiran Kumar,21,Male,Civil,2,73,69,65,40,85,Bangalore,2023,CE
7,1008,Divya Singh,19,Female,Computer Science,2,88,91,84,93,96,Lucknow,2023,CS
8,1009,Rahul Mishra,20,Male,Mechanical,2,62,71,58,45,68,Varanasi,2023,ME
9,1010,Ananya Das,19,Female,Computer Science,2,95,89,90,97,98,Kolkata,2023,CS


In [ ]:
# =========================================================
# INNER JOIN
# Only matching records from both tables
# =========================================================

query_inner = """
SELECT
s.name,
s.math_score,
d.dept_name,
d.hod,
d.established

FROM students AS s

INNER JOIN departments AS d
ON s.dept_code = d.dept_code

ORDER BY s.math_score DESC
LIMIT 8
"""

run_query(query_inner)

# =========================================================
# LEFT JOIN
# All records from LEFT table + matching from RIGHT
# =========================================================

query_left = """
SELECT
s.name,
s.department,
s.math_score,
d.dept_name,
d.hod

FROM students AS s

LEFT JOIN departments AS d
ON s.dept_code = d.dept_code
"""

run_query(query_left)

# =========================================================
# DISPLAY ALL STUDENTS
# =========================================================

q7 = "SELECT * FROM students"

run_query(q7)

          name  math_score        dept_name       hod  established
    Ananya Das          95 Computer Science Dr sharma         1997
   Tanvi Mehta          93 Computer Science Dr sharma         1997
    Arjun Nair          92 Computer Science Dr sharma         1997
Akanksha Yadav          91 Computer Science Dr sharma         1997
Swati Kulkarni          90 Computer Science Dr sharma         1997
   Divya Singh          88 Computer Science Dr sharma         1997
  Ritu Agarwal          87      Electronics   Dr Guru         1996
     Amit Bose          86 Computer Science Dr sharma         1997
          name       department  math_score        dept_name       hod
  Aarav Sharma Computer Science          85 Computer Science Dr sharma
   Priya Patel Computer Science          76 Computer Science Dr sharma
   Rohit Verma      Electronics          65      Electronics   Dr Guru
   Sneha Reddy       Mechanical          70       Mechanical Dr Hamlin
    Arjun Nair Computer Science          9

,student_id,name,age,gender,department,semester,math_score,science_score,english_score,programming_score,attendance_percentage,city,admission_year,dept_code
0,1001,Aarav Sharma,19,Male,Computer Science,2,85,78,72,91,92,Mumbai,2023,CS
1,1002,Priya Patel,20,Female,Computer Science,2,76,82,88,79,87,Ahmedabad,2023,CS
2,1003,Rohit Verma,19,Male,Electronics,2,65,74,61,55,78,Delhi,2023,EC
3,1004,Sneha Reddy,20,Female,Mechanical,2,70,80,75,48,95,Hyderabad,2023,ME
4,1005,Arjun Nair,19,Male,Computer Science,2,92,88,81,95,90,Kochi,2023,CS
5,1006,Meera Joshi,20,Female,Electronics,2,58,66,70,52,72,Pune,2023,EC
6,1007,Kiran Kumar,21,Male,Civil,2,73,69,65,40,85,Bangalore,2023,CE
7,1008,Divya Singh,19,Female,Computer Science,2,88,91,84,93,96,Lucknow,2023,CS
8,1009,Rahul Mishra,20,Male,Mechanical,2,62,71,58,45,68,Varanasi,2023,ME
9,1010,Ananya Das,19,Female,Computer Science,2,95,89,90,97,98,Kolkata,2023,CS


In [ ]:
chart1="""SELECT department,ROUND(AVG(math_score),2) AS avg_math FROM students GROUP BY department ORDER BY avg_math DESC"""
chart1_dat=pd.read_sql_query(chart1, conn)

plt.figure(figsize=(10, 6))
plt.bar(chart1_dat['department'], chart1_dat['avg_math'], color='skyblue')
plt.xlabel('Department')
plt.ylabel('Average Math Score')
plt.title('Average Math Score by Department')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
unique_departments = df['department'].unique()

for dept in unique_departments:
    dept_df = df[df['department'] == dept]

    # Filter by gender for the current department
    male_scores = dept_df[dept_df['gender'] == 'Male']['programming_score']
    female_scores = dept_df[dept_df['gender'] == 'Female']['programming_score']

    plt.figure(figsize=(10, 6))

    # Plot histogram for Male students
    if not male_scores.empty:
        plt.hist(male_scores, bins=range(0, 101, 10), alpha=0.6, label='Male', color='blue', edgecolor='black')

    # Plot histogram for Female students
    if not female_scores.empty:
        plt.hist(female_scores, bins=range(0, 101, 10), alpha=0.6, label='Female', color='red', edgecolor='black')

    plt.xlabel('Programming Score')
    plt.ylabel('Number of Students')
    plt.title(f'Programming Score Distribution by Gender in {dept}')
    plt.legend()
    plt.grid(axis='y', alpha=0.75)
    plt.xticks(range(0, 101, 10))
    plt.tight_layout()
    plt.show()